In [4]:
import sys

omix_path = "/workspace/1AlgoG3/python_packages/multiomix"
fileverse_path = "/workspace/1AlgoG3/python_packages/fileverse17"
statomix_path = "/workspace/1AlgoG3/projects/germinal_centre/statomix"

sys.path.append(omix_path)
sys.path.append(statomix_path)
sys.path.append(fileverse_path)

In [7]:
import pandas as pd
from pathlib import Path
from dataclasses import dataclass

from fileverse.logger import Logger
from fileverse.formats.yaml import BaseYAML
from fileverse.formats.zarr import BaseZARR
from fileverse.formats.excel import BaseExcel

from statomix.project.project import Project
from statomix.pipelines.cleaner.col.col_semantic_rules import DataTypes

In [8]:
path_N0 = "/workspace/1AlgoG3/projects/germinal_centre/gc_w_clinical_csv/N0_Updated.csv"
path_OCAT = "/workspace/1AlgoG3/projects/germinal_centre/gc_w_clinical_csv/OCAT/patient_wise_stats_with_clinical_data_source_updated.csv"
path_Priyanka = "/workspace/1AlgoG3/projects/germinal_centre/gc_w_clinical_csv/Priyanka_Thesis/patient_wise_stats_with_clinical_data-UPDATED.xlsx"

df_N0 = pd.read_csv(path_N0)
df_OCAT = pd.read_csv(path_OCAT)
df_Priyanka = pd.read_excel(path_Priyanka)

common_pids = set(df_N0['Patient ID']).intersection(set(df_Priyanka['Patient ID']))

df_Priyanka = df_Priyanka[~df_Priyanka["Patient ID"].isin(common_pids)]

In [9]:
project_name="Germinal Center (Statomix Trial 2)"

In [10]:
project = Project(project_name=project_name)

INFO (2026-08-02 16:13:49)
Discovered and loaded existing dataset: 'N0'
INFO (2026-08-02 16:13:49)
Discovered and loaded existing dataset: 'OCAT'
INFO (2026-08-02 16:13:49)
Discovered and loaded existing dataset: 'Priyanka'


In [11]:
df = df_Priyanka
dataset_name = "Priyanka"

project.add_dataset(df=df, dataset_name=dataset_name)

WARNING (2026-08-02 16:13:49)
Dataset 'Priyanka' already exists in this project. Please choose a unique name or delete the existing dataset.
Note: the provided DataFrame is NOT identical to the saved DataFrame.
DEBUG (2026-08-02 16:13:49)
DataFrame.index are different

DataFrame.index values are different (95.90164 %)
[left]:  RangeIndex(start=0, stop=122, step=1)
[right]: Index([  0,   1,   2,   3,   4,   6,   7,   8,   9,  10,
       ...
       115, 116, 117, 118, 119, 120, 121, 122, 123, 124],
      dtype='int64', length=122)
At positional index 5, first diff: 5 != 6


# Cleaner Pipeline

In [12]:
version = 1
config_version = 1

In [13]:
dataset = project.datasets[dataset_name]

In [14]:
cleaner = dataset.cleaner

In [15]:
cleaner.create_col_report(version=version, version_name='default')

INFO (2026-08-02 16:13:49)
Column report already exists for version 1. Set create_new=True to create a new one.


In [16]:
cleaner.create_col_edit_schema(version=version)

INFO (2026-08-02 16:13:50)
Column edit schema already exists for version:1.


In [17]:
cleaner.create_cat_meta_report(version=version, config_version=config_version, config_name='default', create_new=False)

INFO (2026-08-02 16:13:50)
Categorical metadata report already exists for version:1 and config_version:1


In [18]:
cleaner.create_cat_meta_edit_schema(version=version, config_version=config_version)

INFO (2026-08-02 16:13:50)
Categorical metadata edit schema already exists for version: 1 and config_version:1


In [19]:
cleaner.create_surv_meta_report(version=version, config_version=config_version)

INFO (2026-08-02 16:13:50)
Survival metadata report already exists for version: 1 and config_version:1


In [20]:
cleaner.create_surv_meta_edit_schema(version=version,config_version=config_version)

INFO (2026-08-02 16:13:50)
Surival meta data already exists for version:1 and config_version:1


In [21]:
cleaner.create_surv_cat_meta_report(version=version, config_version=config_version)

INFO (2026-08-02 16:13:51)
Survival categorical metadata report already exists for version: 1 and config_version:1


In [22]:
cleaner.create_surv_cat_meta_edit_schema(version=version, config_version=config_version)

INFO (2026-08-02 16:13:51)
Survival categorical metadata edit schema already exists for version: 1 and config_version:1


In [23]:
cleaner.create_curated_data(version=version, config_version=config_version)

INFO (2026-08-02 16:13:51)
Curated data already exists for version:1 and config_version:1


In [24]:
dataset.configure_analyzer(version=version, config_version=config_version)

INFO (2026-08-02 16:13:51)
Analyzer data already exists for version:1 and config_version:1


In [25]:
dataset.analyzer.create_summary_report(version=version, config_version=config_version)

INFO (2026-08-02 16:13:51)
Summary report already exists.


# Overall Summary

In [29]:
project.create_datatype_map_overview(version=version, config_version=config_version)

Already exists


# Project Level Analysis

In [57]:
from great_tables import GT

In [28]:
project.analyzer.create_analysis_config(
    project=project, 
    version=None, 
    config_version=None, 
    analysis_name="Experiment 1", 
    create_new=False
)

In [30]:
analysis_config_df = pd.read_excel('analysis_config_version1_curated.xlsx')

In [76]:
tables = []
uids = analysis_config_df['UID'].dropna().unique()

for uid in uids:
    uid_df =(analysis_config_df[analysis_config_df['UID']==uid]).copy()

    num_df = uid_df[uid_df['Datatype'] == 'Numerical']
    num_df = num_df.dropna(axis=1)
    
    cat_df = uid_df[uid_df['Datatype'] == 'Categorical']
    cat_df = cat_df.dropna(axis=1)
    
    column_name_cols = [col for col in num_df.columns if col.startswith("Column Name")]

    req_df_dict = {}
    for _, row in num_df.iterrows():
        dataset = project.datasets[row['Dataset']]
        group_analyzer = dataset.analyzer._get_group_analyzer(version=None, config_version=None)
    
        num_summary_df = group_analyzer.get_num_summary_df()
    
        req_num_df = num_summary_df.filter(items=list(row[column_name_cols]), axis=0)
        req_num_df = req_num_df.round(2).reset_index(names="col_name")
        
        req_num_df["Mean (SD)"] = req_num_df["mean"].astype(str) + "(" + req_num_df["std"].astype(str) + ")"
        req_num_df["Median (IQR)"] = req_num_df["median"].astype(str) + "(" + req_num_df["iqr"].astype(str) + ")"
        
        req_num_df = req_num_df[["col_name", "Mean (SD)", "Median (IQR)"]]
    
        req_df_dict[row['Dataset']] = req_num_df

    summary_dict = req_df_dict

    summary_table = create_summary_table(
        summary_dict,
        title="Comparison of Summary Statistics",
    )
    
    tables.append(summary_table)

ValueError: summary_dict cannot be empty.

In [82]:
tables[2]

GT(_tbl_data=Empty DataFrame
Columns: [Measure, n0_mean_sd, n0_median_iqr, ocat_mean_sd, ocat_median_iqr, priyanka_mean_sd, priyanka_median_iqr]
Index: [], _body=<great_tables._gt_data.Body object at 0x7f66ec5873b0>, _boxhead=Boxhead([ColInfo(var='Measure', type=<ColInfoTypeEnum.stub: 2>, column_label='Measure', column_align='right', column_width=None), ColInfo(var='n0_mean_sd', type=<ColInfoTypeEnum.default: 1>, column_label='Mean (SD)', column_align='center', column_width=None), ColInfo(var='n0_median_iqr', type=<ColInfoTypeEnum.default: 1>, column_label='Median (IQR)', column_align='center', column_width=None), ColInfo(var='ocat_mean_sd', type=<ColInfoTypeEnum.default: 1>, column_label='Mean (SD)', column_align='center', column_width=None), ColInfo(var='ocat_median_iqr', type=<ColInfoTypeEnum.default: 1>, column_label='Median (IQR)', column_align='center', column_width=None), ColInfo(var='priyanka_mean_sd', type=<ColInfoTypeEnum.default: 1>, column_label='Mean (SD)', column_align='center', column_width=None), ColInfo(var='priyanka_median_iqr', type=<ColInfoTypeEnum.default: 1>, column_label='Median (IQR)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f66aeead040>, _spanners=Spanners([SpannerInfo(spanner_id='N0', spanner_level=0, spanner_label='N0', spanner_units=None, spanner_pattern=None, vars=['n0_mean_sd', 'n0_median_iqr'], built=None), SpannerInfo(spanner_id='OCAT', spanner_level=0, spanner_label='OCAT', spanner_units=None, spanner_pattern=None, vars=['ocat_mean_sd', 'ocat_median_iqr'], built=None), SpannerInfo(spanner_id='Priyanka', spanner_level=0, spanner_label='Priyanka', spanner_units=None, spanner_pattern=None, vars=['priyanka_mean_sd', 'priyanka_median_iqr'], built=None)]), _heading=Heading(title='Comparison of Summary Statistics', subtitle=None, preheader=None), _stubhead='Measure', _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7f66ec58e510>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='auto'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='14px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=Options

In [61]:
import re
import pandas as pd
from great_tables import GT


def create_summary_table(
    summary_dict: dict[str, pd.DataFrame],
    title: str = "Summary Statistics",
) -> GT:
    """
    Create a side-by-side Great Tables summary from a dictionary
    of DataFrames.

    Expected columns in each DataFrame:
        - col_name
        - Mean (SD)
        - Median (IQR)
    """

    if not summary_dict:
        raise ValueError("summary_dict cannot be empty.")

    required_columns = {"col_name", "Mean (SD)", "Median (IQR)"}

    wide_df = None
    spanner_columns = []
    display_labels = {}
    all_value_columns = []

    for group_name, df in summary_dict.items():

        # Remove accidental spaces from column names
        current_df = df.copy()
        current_df.columns = current_df.columns.str.strip()

        missing_columns = required_columns - set(current_df.columns)

        if missing_columns:
            raise ValueError(
                f"{group_name} is missing columns: "
                f"{sorted(missing_columns)}"
            )

        current_df = current_df[
            ["col_name", "Mean (SD)", "Median (IQR)"]
        ].copy()

        # Normalize apostrophes:
        # "Total GC’s" and "Total GC's" become the same row
        current_df["Measure"] = (
            current_df["col_name"]
            .astype(str)
            .str.strip()
            .str.translate(
                str.maketrans(
                    {
                        "’": "'",
                        "‘": "'",
                        "′": "'",
                    }
                )
            )
        )

        # Create safe internal column names
        group_id = re.sub(
            pattern=r"[^A-Za-z0-9]+",
            repl="_",
            string=str(group_name),
        ).strip("_").lower()

        mean_column = f"{group_id}_mean_sd"
        median_column = f"{group_id}_median_iqr"

        current_df = current_df[
            ["Measure", "Mean (SD)", "Median (IQR)"]
        ].rename(
            columns={
                "Mean (SD)": mean_column,
                "Median (IQR)": median_column,
            }
        )

        # Merge groups side by side
        if wide_df is None:
            wide_df = current_df
        else:
            wide_df = wide_df.merge(
                current_df,
                on="Measure",
                how="outer",
                sort=False,
                validate="one_to_one",
            )

        spanner_columns.append(
            (str(group_name), [mean_column, median_column])
        )

        display_labels[mean_column] = "Mean (SD)"
        display_labels[median_column] = "Median (IQR)"

        all_value_columns.extend(
            [mean_column, median_column]
        )

    # Construct the Great Tables object
    summary_table = (
        GT(
            wide_df,
            rowname_col="Measure",
        )
        .tab_header(
            title=title,
        )
        .tab_stubhead(
            label="Measure",
        )
        .cols_label(
            **display_labels,
        )
        .cols_align(
            align="center",
            columns=all_value_columns,
        )
        .tab_options(
            table_width="100%",
            table_layout="auto",
            table_font_size="14px",
            data_row_padding="7px",
            column_labels_padding="8px",
            column_labels_font_weight="bold",
            stub_font_weight="bold",
        )
    )

    # Add N0, OCAT and Priyanka headings
    for group_name, columns in spanner_columns:
        summary_table = summary_table.tab_spanner(
            label=group_name,
            columns=columns,
        )

    return summary_table

GT(_tbl_data=                        Measure      n0_mean_sd  n0_median_iqr  \
0           Average GC per Node      5.67(3.53)      5.0(4.27)   
1  Node Area occupied by GC (%)      1.87(1.34)     1.67(1.04)   
2                 Nodes with GC     22.88(10.4)    22.0(11.25)   
3              Nodes without GC     16.5(11.77)     14.0(14.0)   
4                    Total GC's  208.57(133.13)  185.0(173.25)   
5                   Total Nodes    39.38(19.37)    36.0(20.25)   
6                  Total Slides      9.24(3.97)      9.0(4.25)   

     ocat_mean_sd ocat_median_iqr priyanka_mean_sd priyanka_median_iqr  
0    15.07(13.62)    11.06(11.13)       7.68(5.38)          5.97(6.53)  
1       2.4(1.57)      2.07(1.88)       1.97(1.21)          1.63(1.39)  
2    14.16(10.65)      11.0(11.0)     21.06(13.65)          19.5(16.0)  
3      4.33(5.27)        3.0(5.0)      13.34(9.95)         11.0(12.75)  
4  188.08(184.68)   130.5(177.25)    242.16(213.9)       180.0(176.25)  
5    18.49(14.38)      14.0(15.0)      34.4(21.94)          30.0(25.0)  
6      4.48(3.02)        4.0(4.0)       7.84(4.69)           7.0(5.75)  , _body=<great_tables._gt_data.Body object at 0x7f66aebc1ca0>, _boxhead=Boxhead([ColInfo(var='Measure', type=<ColInfoTypeEnum.stub: 2>, column_label='Measure', column_align='left', column_width=None), ColInfo(var='n0_mean_sd', type=<ColInfoTypeEnum.default: 1>, column_label='Mean (SD)', column_align='center', column_width=None), ColInfo(var='n0_median_iqr', type=<ColInfoTypeEnum.default: 1>, column_label='Median (IQR)', column_align='center', column_width=None), ColInfo(var='ocat_mean_sd', type=<ColInfoTypeEnum.default: 1>, column_label='Mean (SD)', column_align='center', column_width=None), ColInfo(var='ocat_median_iqr', type=<ColInfoTypeEnum.default: 1>, column_label='Median (IQR)', column_align='center', column_width=None), ColInfo(var='priyanka_mean_sd', type=<ColInfoTypeEnum.default: 1>, column_label='Mean (SD)', column_align='center', column_width=None), ColInfo(var='priyanka_median_iqr', type=<ColInfoTypeEnum.default: 1>, column_label='Median (IQR)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f66aebc0c20>, _spanners=Spanners([SpannerInfo(spanner_id='N0', spanner_level=0, spanner_label='N0', spanner_units=None, spanner_pattern=None, vars=['n0_mean_sd', 'n0_median_iqr'], built=None), SpannerInfo(spanner_id='OCAT', spanner_level=0, spanner_label='OCAT', spanner_units=None, spanner_pattern=None, vars=['ocat_mean_sd', 'ocat_median_iqr'], built=None), SpannerInfo(spanner_id='Priyanka', spanner_level=0, spanner_label='Priyanka', spanner_units=None, spanner_pattern=None, vars=['priyanka_mean_sd', 'priyanka_median_iqr'], built=None)]), _heading=Heading(title='Comparison of Summary Statistics', subtitle=None, preheader=None), _stubhead='Measure', _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7f66aebcdd00>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='auto'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', val